# 02 UV Comparison — Gold QA(可视化专用)
baseline/TD packed UV + 相同 raw atlas budget 的 rebake 对比。
此处 packing/rebake 仅用于 QA, 不影响 dataset acceptance。
依赖 PARTUV_ROOT teacher checkout(gold_evaluator)。

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import matplotlib.pyplot as plt
from meshuv.data.dataset import CleanDataset
from meshuv.visualization import pipeline_views as V

DATASET = "processed/clean_v1"      # 相对 MESHUV_DATA_ROOT(env 可覆盖)
root = V.resolve(DATASET)
ds = CleanDataset(root, expose_diagnostics=True) if root else None
print("objects:", ds and len(ds))

In [ ]:
from meshuv.evaluation.gold_evaluator import compare_uniform_td
UID = None; RANK = None; SEED = 0
i = V.pick(ds, uid=UID, rank=RANK, seed=SEED)
item = ds[i]
r = compare_uniform_td(root, item)   # 相同 raw atlas budget R×R
if r["status"] == "OK":
    fig = plt.figure(figsize=(16, 8))
    for k, (key, title) in enumerate([("uniform_uv", "baseline packed UV"),
                                      ("td_uv", "TD packed UV"),
                                      ("uniform_tex", "uniform rebake"),
                                      ("td_tex", "TD rebake"),
                                      ("diff", "render |diff|")]):
        ax = fig.add_subplot(2, 3, k + 1)
        r["draw"][key](ax)
    ax = fig.add_subplot(2, 3, 6); ax.set_axis_off()
    ax.text(0.02, 0.9, r["metrics_text"], family="monospace", fontsize=10,
            va="top", transform=ax.transAxes)
    plt.tight_layout(); plt.show()
else:
    print("QA packing 失败:", r)